In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
import time
import numpy as np
import mlflow

class LLMHierarchyCOL:
    def __init__(self, llm_name: str):
        # initialize LLM once
        self.llm = OllamaLLM(model=llm_name, temperature=0.1)

        # --- detailed summary chain ---
        detailed_system = """
You are an expert that generates detailed product descriptions.
You receive a short compact product description. The detailed product description will be later used to extract Named Entities or Key Words to build a taxonomy

Side information: the product description includes ingredients (e.g almond oil), usage information (e. g. silky hair), package dimensions etc
Be creative. Make it about a paragraph long

Rules:
- Don't include explanations or notes
- Don't hallucinate
- Only add valuable information to the product
- Prioritize sizing or packaging dimensions less, but still include them
"""
        detailed_user = "This is the description of the product. {product_description}"
        detail_tpl = ChatPromptTemplate(
            messages=[("system", detailed_system), ("human", detailed_user)],
            input_variables=["product_description"]
        )
        self.detail_chain = detail_tpl | self.llm

        # --- generate taxonomy chain ---
        gen_system = """
You are a helpful assistant and expert in constructing a taxonomy from a given concept.
You will build iteratively a taxonomy with a root concept and from the taxonomy in the previous step and the entity list.

The format of the generated taxonomy is:
1. Parent Concept
1.1 Child Concept.
1.1.1 Sub Concept.
1.2 Child Concept.
1.2.1 Sub Concept.


Critical rules
- DO NOT ADD ANY COMMENTS OR EXPLANATIONS
- THERE IS ONE AND ONLY ONE ROOT NODE OF THE TAXONOMY
- ALL ENTITIES FROM THE LIST MUST APPEAR IN THE TAXONOMY IF FEASIBLE
- YOU ARE ALLOWED TO ADD OTHER ENTITIES IF THEY MAKE SENSE, DON'T HALLUCINATE
- MERGE ENTITIES WITH THE SAME NAME
- EXCLUDE ENTITIES WITH NO INFORMATION DETAIL FOR THE PARENT CONCEPT
- YOU ARE ALLOWED TO ADD OTHER ENTITIES IF THEY MAKE SENSE, DON'T HALLUCINATE
- ONLY RETURN THE TAXONOMY
"""
        gen_user = """
The root entity is {root_concept}, the taxonomy in the current step is:
{taxonomy}

The entity list for this step contains:
{list_entities}
"""
        gen_tpl = ChatPromptTemplate(
            messages=[("system", gen_system), ("user", gen_user)],
            input_variables=["root_concept", "taxonomy", "list_entities"]
        )
        self.gen_chain = gen_tpl | self.llm

        # --- update taxonomy chain ---
        update_system = """
You are a helpful assistant and expert in updating a given hierarchical taxonomy with a root concept.
Review the given taxonomy from the previous step.

If the taxonomy contains unreasonable or wrong relations, update them to make sense.
You may relocate child nodes to new parents if child nodes have low affinity to the parent concept and add nodes if necessary.

Example: 
1.1 Playstation
1.1.1 Playstation 2
1.1.2 Playstation 3
1.1.3 Xbox 360
1.2 Xbox
1.2.1 Xbox One

to 

1.1 Playstation
1.1.1 Playstation 2
1.1.2 Playstation 3
1.2 Xbox
1.2.1 Xbox One
1.2.2 Xbox 360



The format of the generated taxonomy is:
1. Parent Concept
1.1 Child Concept.
1.1.1 Sub Concept.
1.2 Child Concept.
1.2.1 Sub Concept.
Do not change any entity names when building the taxonomy.

Critical rules
- DO NOT ADD ANY COMMENTS OR EXPLANATIONS
- THERE IS ONE AND ONLY ONE ROOT NODE
- ALL ENTITIES MUST APPEAR IF FEASIBLE
- YOU ARE ALLOWED TO ADD OTHER ENTITIES IF THEY MAKE SENSE, DON'T HALLUCINATE
- MERGE ENTITIES WITH THE SAME NAME
- EXCLUDE ENTITIES WITH NO DETAIL
- KEEP THE NUMBERING FORMAT
- ONLY RETURN THE TAXONOMY
"""
        update_user = """
The root entity is {root_concept}.
The entity list contains: {list_entities}

Current taxonomy:
{taxonomy}
"""
        update_tpl = ChatPromptTemplate(
            messages=[("system", update_system), ("user", update_user)],
            input_variables=["root_concept", "list_entities", "taxonomy"]
        )
        self.update_chain = update_tpl | self.llm

        # --- review taxonomy chain ---
        review_system = """
You are an expert for validating a generated hierarchical taxonomy.
Your task is to check if the taxonomy suits the product description.

If it's unsuitable, return FALSE.
If it's suitable, return TRUE.

Critical rules:
- Do not include any explanations or notes
- ONLY RETURN THE BOOLEAN
- Do not hallucinate
- Validate carefully

Output Format:
BOOLEAN
"""
        review_user = """
The root entity is {root_concept}.
Entity list: {list_entities}

Taxonomy to review:
{taxonomy}

Product description:
{product_description}
"""
        review_tpl = ChatPromptTemplate(
            messages=[("system", review_system), ("user", review_user)],
            input_variables=["root_concept", "list_entities", "taxonomy", "product_description"]
        )
        self.review_chain = review_tpl | self.llm

        # --- embedding & keyword models ---
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embed_kw_model = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
        self.kw_model = KeyBERT(model=self.embed_kw_model)

    def generate_details(self, short_summary: str) -> str:
        return self.detail_chain.invoke({"product_description": short_summary})

    def get_keywords(self, detailed_summary: str) -> list:
        raw = self.kw_model.extract_keywords(
            detailed_summary,
            keyphrase_ngram_range=(1, 2),
            top_n=5,
            stop_words="english",
            use_maxsum=True
        )
        # return unique keywords
        return list({kw for kw, _ in raw})

    def generate_tax(self, root_concept: str, list_keywords: list, taxonomy: str) -> str:
        return self.gen_chain.invoke({
            "root_concept": root_concept,
            "taxonomy": taxonomy,
            "list_entities": list_keywords
        })

    def update_tax(self, taxonomy: str, root_concept: str, list_keywords: list) -> str:
        return self.update_chain.invoke({
            "root_concept": root_concept,
            "list_entities": list_keywords,
            "taxonomy": taxonomy
        })

    def review(self, detailed_summary: str, taxonomy: str, root_concept: str, list_keywords: list) -> bool:
        out = self.review_chain.invoke({
            "root_concept": root_concept,
            "list_entities": list_keywords,
            "taxonomy": taxonomy,
            "product_description": detailed_summary
        })
        return out.strip().upper() == "TRUE"

    def taxonomy_to_triples(self, taxonomy_text: str) -> pd.DataFrame:
        lines = taxonomy_text.strip().split("\n")
        triples = []
        hierarchy = {}
        for line in lines:
            if not line.strip():
                continue
            parts = line.strip().split(" ", 1)
            if len(parts) < 2:
                continue
            level_num, concept = parts
            level_num = level_num.rstrip(".")
            hierarchy[level_num] = concept
            if "." in level_num:
                parent_level = ".".join(level_num.split(".")[:-1])
                if parent_level in hierarchy:
                    triples.append({
                        "head": hierarchy[parent_level],
                        "relation": "is parent of",
                        "tail": concept
                    })
        return pd.DataFrame(triples, columns=["head", "relation", "tail"])

    def linkage_asin_to_taxonomy(
        self,
        triples_df: pd.DataFrame,
        dict_asin_keywords: dict,
        sim_thresh: float = 0.5
    ) -> pd.DataFrame:
        heads = triples_df["head"].tolist()
        tails = triples_df["tail"].tolist()
        H = self.embed_model.encode(heads)  # (n, d)
        T = self.embed_model.encode(tails)  # (n, d)

        new_links = []
        for asin, kws in dict_asin_keywords.items():
            K = self.embed_model.encode(kws)  # (k, d)
            sim_h = K @ H.T                  # (k, n)
            sim_t = K @ T.T                  # (k, n)
            # find all keyword→taxonomy matches over threshold
            idxs_h = np.argwhere(sim_h > sim_thresh)
            idxs_t = np.argwhere(sim_t > sim_thresh)
            for i, j in idxs_h:
                new_links.append({"head": asin, "relation": "related to", "tail": heads[j]})
            for i, j in idxs_t:
                new_links.append({"head": asin, "relation": "related to", "tail": tails[j]})

        if new_links:
            out = pd.concat([triples_df, pd.DataFrame(new_links)], ignore_index=True)
            return out.drop_duplicates().reset_index(drop=True)
        return triples_df


if __name__ == "__main__":
    # Hyperparameters
    DATASETS = ["Books", "MovieLens-1M", "Video_Games", "Last-FM"]
    DATASET = DATASETS[1]  # e.g. "All_Beauty"
    DIR_NAME = "amazon-"
    LLM_MODEL = "gemma3"
    SENTENCE_TRANSFORMER = "all-MiniLM-L6-v2"
    ROOT_CONCEPT = "Product details"
    EXPERIMENT_NAME = "Compairson of LLMs"
    EXPERIMENT_NAME = "LLM-Col MovieLens-1M"
    EXPERIMENT_NAME = "LLM-Col local Batch sizes"


    # Paths
    current_dir = os.getcwd()
    data_path = os.path.join(
        current_dir,
        "data",
        "preprocessed",
        f"{DIR_NAME}{DATASET}",
        "metadata_filtered_df.csv"
    )

    # Load data
    metadata_filtered_df = pd.read_csv("/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/preprocessed/amazon-Video_games/metadata_filtered_df.csv")
    #metadata_filtered_df = pd.read_csv("/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/preprocessed/MovieLens-1M/metadata_df.csv")
    # MLFlow
    os.environ['MLFLOW_TRACKING_USERNAME'] = "jonas.limniatis2"
    os.environ['MLFLOW_TRACKING_PASSWORD'] = "4563359ec5513f3621677b9729db8a4c617e07eb"
    os.environ['MLFLOW_TRACKING_PROJECTNAME'] = "LLM-Col"

    mlflow.set_tracking_uri(f'https://dagshub.com/' + os.environ['MLFLOW_TRACKING_USERNAME']
                            + '/' + os.environ['MLFLOW_TRACKING_PROJECTNAME'] + '.mlflow')

    mlflow.set_experiment(EXPERIMENT_NAME)
    mlflow.start_run(run_name=f"Batch size 10")
    mlflow.set_tag("dataset", DATASET)
    mlflow.set_tag("llm", LLM_MODEL)
    mlflow.set_tag("Similarity Threshhold", "0.5")
    mlflow.set_tag("Batch size", "10")
    # Enabling autolog for LangChain will enable trace logging.
    mlflow.langchain.autolog()


    llm_hierarchy = LLMHierarchyCOL(LLM_MODEL)

    start = time.time()


    # 1) Generate detailed summaries (streaming to save memory)
   # for idx, row in metadata_filtered_df.iterrows():
        #print(f"Detailing row {idx}")
       # metadata_filtered_df.at[idx, "detailed_summary"] = (
       #     llm_hierarchy.generate_details(row.source_text)
      #  )

    # Or directly (MLflow >= 1.11.0):
    #mlflow.log_dict(metadata_filtered_df.to_dict(orient="records"), "data/metadata_filtered_df.json")


    # 2) Batch‐wise taxonomy building

    BATCH_SIZE = 5
    BATCH_SIZE = 10
    dict_asin_keywords = {}
    taxonomy_text = ""

    for i in range(0, len(metadata_filtered_df), BATCH_SIZE):

        print(f"Processing batch {i} to {i + BATCH_SIZE}")
        batch = metadata_filtered_df.iloc[i : i + BATCH_SIZE]
        batch_kw_lists = []
        for _, item in batch.iterrows():
            kws = llm_hierarchy.get_keywords(item.detailed_summary)
            batch_kw_lists.append(kws)
            dict_asin_keywords[item.parent_asin] = kws

        # generate → update → review
        taxonomy_text = llm_hierarchy.generate_tax(ROOT_CONCEPT, batch_kw_lists, taxonomy_text)
        taxonomy_text = llm_hierarchy.update_tax(taxonomy_text, ROOT_CONCEPT, batch_kw_lists)
        ok = llm_hierarchy.review(batch.iloc[0].detailed_summary, taxonomy_text, ROOT_CONCEPT, batch_kw_lists)
        if not ok:
            taxonomy_text = llm_hierarchy.generate_tax(ROOT_CONCEPT, batch_kw_lists, taxonomy_text)
            ok = llm_hierarchy.review(batch.iloc[0].detailed_summary, taxonomy_text, ROOT_CONCEPT, batch_kw_lists)

        # transform → link → save
        triples_df = llm_hierarchy.taxonomy_to_triples(taxonomy_text)

    mlflow.log_dict(dict_asin_keywords, "data/dict_asin_keywords.json") # log keyword dict
    linked_df = llm_hierarchy.linkage_asin_to_taxonomy(triples_df, dict_asin_keywords)
    mlflow.log_dict(linked_df.to_dict(orient="records"), f"data/{LLM_MODEL}-taxonomy_triples.json") # log triples


    out_path = os.path.join(
          current_dir,
          "data",
          "taxonomy",
          f"{DIR_NAME}{DATASET}",
          f"{LLM_MODEL}_taxonomy_triples_batch_size_10.csv"
      )

    output_path = f'/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/taxonomy/amazon-Video_games/{LLM_MODEL}_taxonomy_batch_size_10_full.csv' # for local
    linked_df.to_csv(output_path, index=False)

    end = time.time()
    print(f"Total runtime: {end - start:.1f}s")
    print(f"Per‐item runtime: {(end - start) / len(metadata_filtered_df):.3f}s")


Processing batch 0 to 10
Processing batch 10 to 20
Processing batch 20 to 30
Processing batch 30 to 40
Processing batch 40 to 50
Processing batch 50 to 60
Processing batch 60 to 70
Processing batch 70 to 80
Processing batch 80 to 90
Processing batch 90 to 100
Processing batch 100 to 110
Processing batch 110 to 120
Processing batch 120 to 130
Processing batch 130 to 140
Processing batch 140 to 150
Processing batch 150 to 160
Processing batch 160 to 170
Processing batch 170 to 180
Processing batch 180 to 190
Processing batch 190 to 200
Processing batch 200 to 210
Processing batch 210 to 220
Processing batch 220 to 230
Processing batch 230 to 240
Processing batch 240 to 250
Processing batch 250 to 260
Processing batch 260 to 270
Processing batch 270 to 280
Processing batch 280 to 290
Processing batch 290 to 300
Processing batch 300 to 310
Processing batch 310 to 320
Processing batch 320 to 330
Processing batch 330 to 340
Processing batch 340 to 350
Processing batch 350 to 360
Processing b

2025/07/19 21:22:23 WARNING mlflow.tracing.export.mlflow: Failed to log trace spans as tag to MLflow backend. Error: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces/220e9c93ddd94eb9922447b61d102216/tags failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces/220e9c93ddd94eb9922447b61d102216/tags (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x43b5b5fd0>: Failed to resolve 'dagshub.com' ([Errno 8] nodename nor servname provided, or not known)")). For full traceback, set logging level to debug.
2025/07/19 23:30:50 WARNING mlflow.tracing.export.mlflow: Failed to log trace to MLflow backend. Error: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow-artifacts/artifacts/4ad16bd1f06247ddadfc61c1809bf460/traces/220e9c93ddd94eb9922447b61d102216/artifacts/traces.json failed with exception HTT

Processing batch 540 to 550


2025/07/20 01:11:11 WARNING mlflow.tracking.client: Failed to start trace RunnableSequence: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x43b5b6870>: Failed to resolve 'dagshub.com' ([Errno 8] nodename nor servname provided, or not known)")). For full traceback, set logging level to debug.
2025/07/20 02:37:04 WARNING mlflow.tracking.client: Failed to start trace RunnableSequence: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x43b5b71

Processing batch 550 to 560


...

Processing batch 8830 to 8840


2025/07/22 09:46:44 WARNING mlflow.tracking.client: Failed to start trace RunnableSequence: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x46ee3f6b0>: Failed to resolve 'dagshub.com' ([Errno 8] nodename nor servname provided, or not known)")). For full traceback, set logging level to debug.
2025/07/22 09:48:37 WARNING mlflow.tracking.client: Failed to start trace RunnableSequence: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x357c8b2

Processing batch 8840 to 8850


2025/07/22 09:51:42 WARNING mlflow.tracking.client: Failed to start trace RunnableSequence: API request to https://dagshub.com/jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /jonas.limniatis2/LLM-Col.mlflow/api/2.0/mlflow/traces (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x4a9cf5490>: Failed to resolve 'dagshub.com' ([Errno 8] nodename nor servname provided, or not known)")). For full traceback, set logging level to debug.


Processing batch 8850 to 8860
Processing batch 8860 to 8870
Processing batch 8870 to 8880
Processing batch 8880 to 8890
Processing batch 8890 to 8900
Processing batch 8900 to 8910
Processing batch 8910 to 8920
Processing batch 8920 to 8930
Processing batch 8930 to 8940
Processing batch 8940 to 8950
Processing batch 8950 to 8960
Processing batch 8960 to 8970
Processing batch 8970 to 8980
Processing batch 8980 to 8990
Processing batch 8990 to 9000
Processing batch 9000 to 9010
Processing batch 9010 to 9020
Processing batch 9020 to 9030
Processing batch 9030 to 9040
Processing batch 9040 to 9050
Processing batch 9050 to 9060
Processing batch 9060 to 9070
Processing batch 9070 to 9080
Processing batch 9080 to 9090
Processing batch 9090 to 9100
Processing batch 9100 to 9110
Processing batch 9110 to 9120
Processing batch 9120 to 9130
Processing batch 9130 to 9140
Processing batch 9140 to 9150
Processing batch 9150 to 9160
Processing batch 9160 to 9170
Processing batch 9170 to 9180
Processing

In [ ]:
import mlflow
mlflow.end_run()